# Module 4: Memory & Chat History

In this notebook, we will explore:
1. **LLM Statelessness**: Verifying why models forget past inputs.
2. **ChatMessageHistory**: Using LangChain container structures to hold messages.
3. **RunnableWithMessageHistory**: Automating the state orchestration loop.
4. **SQLite Persistence**: Saving chat histories to a local database.
5. **Context Trimming**: Pruning history to control token lengths and prevent context window issues.

### Step 1: Initialize Chat Model Connection

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="google/gemma-2-9b-it:free",
    temperature=0.5,
)
print("Model client connected!")

---
## 1. Proving Statelessness

Let's make two consecutive requests to the model to see how it performs without memory.

In [ ]:
from langchain_core.messages import HumanMessage

res1 = model.invoke([HumanMessage(content="Hi, my name is Bob. I live in Toronto.")])
print("Turn 1 Response:", res1.content)

res2 = model.invoke([HumanMessage(content="What is my name and where do I live?")])
print("\nTurn 2 Response:", res2.content)

Because Turn 2 was a completely fresh connection containing no context, the model could not answer. Now, let's manually build a historical list of messages to see the change.

In [ ]:
from langchain_core.messages import AIMessage

context = [
    HumanMessage(content="Hi, my name is Bob. I live in Toronto."),
    AIMessage(content=res1.content), # Feed the model's own response back
    HumanMessage(content="What is my name and where do I live?")
]

res3 = model.invoke(context)
print("Turn 2 (With Manual History) Response:")
print(res3.content)

---
## 2. Using `ChatMessageHistory` and `MessagesPlaceholder`

Instead of managing raw Python lists, we utilize `InMemoryChatMessageHistory` to store conversations. We will inject these message structures into a `ChatPromptTemplate` using `MessagesPlaceholder`.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. Initialize history storage container
history = InMemoryChatMessageHistory()

# 2. Build prompt template containing MessagesPlaceholder
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

print("Prompt messages configured:")
print(chat_prompt.messages)

Let's trace a manual session loop: we load history -> format prompt -> invoke -> save results.

In [ ]:
# Turn 1
question1 = "What is the capital of Japan?"
formatted_messages = chat_prompt.format_messages(
    chat_history=history.messages,
    question=question1
)
reply1 = model.invoke(formatted_messages)

# Update history container with both messages
history.add_user_message(question1)
history.add_ai_message(reply1.content)

print("AI Reply 1:", reply1.content)
print("Current History Length:", len(history.messages))

# Turn 2
question2 = "What is its population?"
formatted_messages = chat_prompt.format_messages(
    chat_history=history.messages,
    question=question2
)
reply2 = model.invoke(formatted_messages)

history.add_user_message(question2)
history.add_ai_message(reply2.content)

print("\nAI Reply 2:", reply2.content)
print("Current History Length:", len(history.messages))

---
## 3. Automating State with `RunnableWithMessageHistory`

Manually updating history lists is tedious. We can wrap our LCEL chain using `RunnableWithMessageHistory` to manage state loading and saving automatically based on session identifiers.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# Session dictionary store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Core chain
base_chain = chat_prompt | model

# Orchestrated chain
conversational_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

print("Wrapped chain created!")

Now we execute queries, passing our `session_id` inside the `config` payload. Let's see how two separate sessions are kept completely isolated.

In [ ]:
# Session A
res_a1 = conversational_chain.invoke(
    {"question": "Hi! I'm Alice. My favorite food is sushi."},
    config={"configurable": {"session_id": "session_alice"}}
)
print("Alice Session Reply:", res_a1.content)

# Session B
res_b1 = conversational_chain.invoke(
    {"question": "Hi! I'm David. I love Italian pasta."},
    config={"configurable": {"session_id": "session_david"}}
)
print("\nDavid Session Reply:", res_b1.content)

# Querying Alice's state again
res_a2 = conversational_chain.invoke(
    {"question": "What is my favorite food?"},
    config={"configurable": {"session_id": "session_alice"}}
)
print("\nAlice Session query-back:", res_a2.content)

---
## 4. Persistent SQLite Database Storage

Instead of losing memory upon application shutdown, we can use `SQLChatMessageHistory` to store conversations in a local database.

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

db_connection = "sqlite:///sqlite_chat_history.db"

def get_sqlite_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string=db_connection
    )

# Wrap base_chain with SQLite database
persistent_chain = RunnableWithMessageHistory(
    base_chain,
    get_sqlite_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

print("Persistent DB conversational chain ready!")

In [ ]:
# Execute dialogue turns
db_res1 = persistent_chain.invoke(
    {"question": "My secret code word is: BANANA-42"},
    config={"configurable": {"session_id": "user_443"}}
)
print("DB Res 1:", db_res1.content)

db_res2 = persistent_chain.invoke(
    {"question": "What was my secret code word?"},
    config={"configurable": {"session_id": "user_443"}}
)
print("\nDB Res 2:", db_res2.content)

Let's read directly from our local SQLite file history using SQLite classes to prove the data is stored in the database tables.

In [ ]:
import sqlite3

conn = sqlite3.connect("sqlite_chat_history.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Database Tables:", cursor.fetchall())

# Retrieve saved records
cursor.execute("SELECT session_id, message FROM message_store LIMIT 4;")
print("\nSaved Database Rows:")
for row in cursor.fetchall():
    print(f"Session: {row[0]} | Message Payload Preview: {row[1][:60]}...")

conn.close()

---
## 5. History Pruning & Trimming

As dialogues grow longer, sending dozens of messages back and forth can exceed the model's maximum context window and cost a significant amount of tokens. We can trim history to limit messages.

We can implement a custom python function inside a `RunnableLambda` to prune the messages array before sending it to the model.

In [ ]:
from langchain_core.runnables import RunnableLambda

# Define a custom trimmer that keeps only the last N messages
def trim_messages_list(chat_history_list, last_n_messages=2):
    # Keep only the last N messages to fit token limits
    return chat_history_list[-last_n_messages:]

# Wrap as RunnableLambda
trimmer = RunnableLambda(lambda x: trim_messages_list(x, last_n_messages=2))

# Create an input dictionary parser that trims the history list
trimmed_chain = (
    RunnablePassthrough.assign(
        chat_history=lambda x: trimmer.invoke(x["chat_history"])
    )
    | chat_prompt
    | model
)

# Test trimmed execution
trim_store = InMemoryChatMessageHistory()
trim_store.add_user_message("My favorite color is green.")
trim_store.add_ai_message("Got it, green!")
trim_store.add_user_message("I work as a civil engineer.")
trim_store.add_ai_message("Civil engineer, noted.")

# Now invoke the chain asking a question about the favorite color
# Since we only keep the last 2 messages (which covers the civil engineer turn),
# the model should NOT remember the favorite color!
res_trimmed = trimmed_chain.invoke({
    "chat_history": trim_store.messages,
    "question": "What is my favorite color?"
})
print("Model Response (With Trimmed History):")
print(res_trimmed.content)